# Mercedes-Benz: offline CPU rerun

## Goal

Run the maintained package against the attached official competition snapshot, independently reconstruct all retained fold-model predictions, and save a portable run bundle. This notebook is an execution entry point; the repository walkthrough explains evaluation and engineering decisions.

## Setup

Attach `mercedes-benz-greener-manufacturing` and the private `kaggle-mercedes-benz-code` dataset generated by `scripts/export_kaggle.py`. Internet and GPU are disabled. The supplied wheel installs without resolving dependencies; hosted scientific package versions are recorded by the run. Native and hosted runs remain separate experiments, and joblib models are evaluated in the environment that produced them.

The full recipe fits three model families on five fixed folds and two model seeds, capped at four CPU threads. No OOF weight search, target clipping, model selection or Kaggle submission occurs here.

In [ ]:
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
COMPETITION = "mercedes-benz-greener-manufacturing"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "4"


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


receipts = []
for path in INPUT_ROOT.rglob("source.json"):
    value = json.loads(path.read_text())
    if value.get("project_id") == "mercedes-benz":
        receipts.append((path, value))
if len(receipts) != 1:
    raise ValueError("Attach exactly one Mercedes-Benz code dataset version")
source_path, code_asset = receipts[0]
asset_root = source_path.parent
wheel = asset_root / code_asset["wheel"]
if wheel.name != code_asset["wheel"] or file_sha(wheel) != code_asset["wheel_sha256"]:
    raise ValueError("Code wheel identity differs from its receipt")
for name, expected in code_asset["config_sha256"].items():
    if Path(name).name != name or file_sha(asset_root / name) != expected:
        raise ValueError("Code configuration checksum mismatch")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--force-reinstall",
        str(wheel),
    ],
    check=True,
    capture_output=True,
    text=True,
)

from mercedes.pipeline import source_identity

installed_source = source_identity()
if installed_source["python_source_sha256"] != code_asset["source"]["python_source_sha256"]:
    raise ValueError("Installed package does not match the code-asset package fingerprint")
print({"wheel_sha256": file_sha(wheel), "package_sha256": installed_source["python_source_sha256"]})


## Steps

### 1. Resolve only the three official CSV inputs

The competition mount may contain CSV files directly or the three original ZIP archives. Copy only the expected files into a writable data directory; do not extract an arbitrary archive tree. The resulting input identities are retained by the training manifest.

In [ ]:
def prepare_inputs(input_root, destination):
    roots = [path for path in input_root.rglob(COMPETITION) if path.is_dir()]
    if len(roots) != 1:
        raise ValueError("Expected one mounted official competition directory")
    competition_root = roots[0]
    destination.mkdir(parents=True, exist_ok=False)
    for name in ("train.csv", "test.csv", "sample_submission.csv"):
        direct = list(competition_root.rglob(name))
        archives = list(competition_root.rglob(name + ".zip"))
        target = destination / name
        if len(direct) == 1 and not archives:
            if direct[0].is_symlink():
                raise ValueError("Input CSV must be a regular file")
            shutil.copyfile(direct[0], target)
        elif not direct and len(archives) == 1:
            with zipfile.ZipFile(archives[0]) as archive:
                members = [
                    member
                    for member in archive.infolist()
                    if PurePosixPath(member.filename).name == name
                ]
                if len(members) != 1:
                    raise ValueError("Expected exactly one named CSV in the archive")
                member = members[0]
                path = PurePosixPath(member.filename)
                if (
                    path.is_absolute()
                    or ".." in path.parts
                    or stat.S_ISLNK(member.external_attr >> 16)
                ):
                    raise ValueError("Unsafe archive member")
                with archive.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
        else:
            raise ValueError("Competition inputs are missing or ambiguous")
    return {path.name: file_sha(path) for path in sorted(destination.iterdir())}


data_dir = WORK_ROOT / "data"
input_hashes = prepare_inputs(INPUT_ROOT, data_dir)
config = asset_root / "config-baseline.json"
run_dir = WORK_ROOT / "run"
if run_dir.exists():
    raise FileExistsError("Choose a fresh working directory; completed run evidence is immutable")
print({"input_sha256": input_hashes, "config_sha256": file_sha(config)})


### 2. Train the fixed recipe and reconstruct every model

Each fold fits its own preprocessing. The equal-weight ensemble is declared before evaluation. CLI logs stay in the working directory; the notebook prints bounded verification summaries rather than raw rows or the complete model manifest.

In [ ]:
def run_cli(arguments, log_name):
    with (WORK_ROOT / log_name).open("w") as log:
        subprocess.run(
            [sys.executable, "-m", "mercedes.cli", *arguments],
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )


run_cli(["validate", "--data-dir", str(data_dir)], "validate.log")
run_cli(
    ["train", "--data-dir", str(data_dir), "--config", str(config), "--run-dir", str(run_dir)],
    "train.log",
)
trained_manifest_sha = file_sha(run_dir / "manifest.json")
run_cli(["evaluate", "--data-dir", str(data_dir), "--run-dir", str(run_dir)], "evaluate.log")
if file_sha(run_dir / "manifest.json") != trained_manifest_sha:
    raise ValueError("Producer manifest changed during evaluation")
reload_receipt = json.loads((WORK_ROOT / "evaluate.log").read_text())
if reload_receipt["status"] != "verified_model_reload":
    raise ValueError("Independent model reconstruction did not complete")
print(reload_receipt)


## Checks

Regenerate all official test predictions from the saved models and check every test ID against the sample submission. A separate code-asset receipt binds the installed wheel to its source/config identity without editing the producer manifest after training.

In [ ]:
run_cli(
    [
        "predict",
        "--run-dir",
        str(run_dir),
        "--data",
        str(data_dir / "test.csv"),
        "--output",
        str(WORK_ROOT / "submission.csv"),
    ],
    "predict.log",
)
import pandas as pd
import numpy as np

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = pd.read_csv(WORK_ROOT / "submission.csv")
if list(submission.columns) != ["ID", "y"]:
    raise ValueError("Submission columns differ")
if not submission.ID.astype(str).equals(sample.ID.astype(str)):
    raise ValueError("Submission IDs or order differ")
if len(submission) != 4209 or not np.isfinite(submission.y.to_numpy()).all():
    raise ValueError("Expected all 4,209 finite official test predictions")
manifest_sha = file_sha(run_dir / "manifest.json")
if manifest_sha != trained_manifest_sha:
    raise ValueError("Producer manifest changed after training")
execution_receipt = {
    "schema_version": 1,
    "project_id": "mercedes-benz",
    "scope": "full-data CPU training, independent same-environment model reload, full test prediction",
    "code_asset": code_asset,
    "source_receipt_sha256": file_sha(source_path),
    "installed_source": installed_source,
    "input_sha256": input_hashes,
    "config_sha256": file_sha(config),
    "producer_manifest_sha256": manifest_sha,
    "producer_manifest_modified": False,
    "model_reload": reload_receipt,
    "submission_rows": len(submission),
    "submission_ids_order_verified": True,
    "submission_sha256": file_sha(WORK_ROOT / "submission.csv"),
}
(WORK_ROOT / "code-asset.json").write_text(json.dumps(execution_receipt, indent=2) + "\n")
with zipfile.ZipFile(
    WORK_ROOT / "run-bundle.zip", "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(run_dir.iterdir()):
        if not path.is_file() or path.is_symlink():
            raise ValueError("Unexpected nonregular producer artifact")
        archive.write(path, "run/" + path.name)
    archive.write(WORK_ROOT / "code-asset.json", "code-asset.json")
if file_sha(run_dir / "manifest.json") != manifest_sha:
    raise ValueError("Producer manifest changed during packaging")
print(
    {
        "submission_rows": len(submission),
        "submission_sha256": execution_receipt["submission_sha256"],
        "bundle_sha256": file_sha(WORK_ROOT / "run-bundle.zip"),
    }
)


## Next steps

Retain the exact Kaggle notebook version, `run-bundle.zip`, `code-asset.json`, logs and `submission.csv` privately. Import verified completed artifacts into the approved tracker only after retrieval. This run does not contact private infrastructure or submit predictions to Kaggle automatically. It does not establish cross-version joblib portability or identical native/hosted training numbers.